In [24]:
import numpy as np, pandas as pd, lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score

DATA_DIR = "/kaggle/input/datasets/me24b1076ayushgupta/online-news"
SEED = 42
SEEDS = [7]
N_FOLDS = 5

tr = pd.read_csv(f"{DATA_DIR}/train.csv", na_values=["NA"])
te = pd.read_csv(f"{DATA_DIR}/test.csv", na_values=["NA"])
tr.columns = tr.columns.str.strip(); te.columns = te.columns.str.strip()

classes = list("ABCDE"); cmap = {c: i for i, c in enumerate(classes)}
y = tr["target_popularity"].map(cmap).values

print(tr.shape, te.shape)
print(tr["target_popularity"].value_counts(normalize=True).sort_index())
na = tr.isna().sum(); print("NA cols:\n", na[na > 0])

(37144, 61) (2500, 60)
target_popularity
A    0.046172
B    0.106208
C    0.194648
D    0.363639
E    0.289333
Name: proportion, dtype: float64
NA cols:
 n_tokens_content              1055
n_unique_tokens               1115
num_hrefs                     1557
num_imgs                      1843
average_token_length          1079
num_keywords                   758
kw_max_avg                    1427
kw_avg_avg                    2269
self_reference_avg_sharess    1921
LDA_02                        1110
global_subjectivity           1126
global_sentiment_polarity     1106
avg_positive_polarity         1519
avg_negative_polarity         1454
title_subjectivity             727
dtype: int64


In [25]:
def fe(d):
    d = d.copy(); eps = 1  # Copying the dataframe and handling zerodivision error
    d["na_count"] = d.isna().sum(axis=1)
    d["tok_per_link"] = d["n_tokens_content"] / (d["num_hrefs"] + eps)
    d["img_per_tok"] = d["num_imgs"] / (d["n_tokens_content"] + eps)
    d["vid_per_tok"] = d["num_videos"] / (d["n_tokens_content"] + eps)
    d["self_link_ratio"] = d["num_self_hrefs"] / (d["num_hrefs"] + eps)
    d["media_total"] = d["num_imgs"] + d["num_videos"]
    d["kw_range_avg"] = d["kw_max_avg"] - d["kw_min_avg"]
    d["kw_avg_x_maxavg"] = d["kw_avg_avg"] * d["kw_max_avg"]
    d["selfref_range"] = d["self_reference_max_shares"] - d["self_reference_min_shares"]
    d["log_selfref_avg"] = np.log1p(d["self_reference_avg_sharess"].clip(lower=0))
    d["log_kw_max_avg"] = np.log1p(d["kw_max_avg"].clip(lower=0))
    d["sent_gap"] = d["avg_positive_polarity"] + d["avg_negative_polarity"]
    d["title_len_ratio"] = d["n_tokens_title"] / (d["n_tokens_content"] + eps)
    lda = [f"LDA_0{i}" for i in range(5)]
    d["lda_max"] = d[lda].max(axis=1)
    d["lda_argmax"] = d[lda].fillna(-1).values.argmax(axis=1)
    chan = [c for c in d.columns if c.startswith("data_channel_is_")]
    d["channel_none"] = (d[chan].sum(axis=1) == 0).astype(int)
    return d

X = fe(tr.drop(columns=["id", "target_popularity"]))
Xt = fe(te.drop(columns=["id"]))
assert list(X.columns) == list(Xt.columns)
print(X.shape, Xt.shape)

(37144, 75) (2500, 75)


In [26]:
def run_lgb(seeds=SEEDS, lr=0.03):
    oof = np.zeros((len(X), 5)); pred = np.zeros((len(Xt), 5))
    for s in seeds:
        params = dict(objective="multiclass", num_class=5, learning_rate=lr,
                      num_leaves=31, min_child_samples=40, feature_fraction=0.7,
                      bagging_fraction=0.8, bagging_freq=1, lambda_l2=5.0,
                      verbose=-1, seed=s)
        skf = StratifiedKFold(N_FOLDS, shuffle=True, random_state=s)
        for f, (a, b) in enumerate(skf.split(X, y)):
            m = lgb.train(params, lgb.Dataset(X.iloc[a], y[a]), 3000,
                          valid_sets=[lgb.Dataset(X.iloc[b], y[b])],
                          callbacks=[lgb.early_stopping(100, verbose=False)])
            oof[b] += m.predict(X.iloc[b], num_iteration=m.best_iteration) / len(seeds)
            pred += m.predict(Xt, num_iteration=m.best_iteration) / (N_FOLDS * len(seeds))
            print(f"seed {s} fold {f}: iters={m.best_iteration}")
    return oof, pred

oof_lgb, pred_lgb = run_lgb()
print("LGB raw OOF macro F1:", f1_score(y, oof_lgb.argmax(1), average="macro"))

seed 7 fold 0: iters=239
seed 7 fold 1: iters=210
seed 7 fold 2: iters=256
seed 7 fold 3: iters=228
seed 7 fold 4: iters=266
LGB raw OOF macro F1: 0.2692222625079359


In [27]:
def tune_weights(P, y, rounds=3):
    w = np.ones(5); grid = np.exp(np.linspace(-1.2, 1.2, 49))
    best = f1_score(y, (P * w).argmax(1), average="macro")
    for _ in range(rounds):
        for c in range(5):
            for g in grid:
                w2 = w.copy(); w2[c] = g
                s = f1_score(y, (P * w2).argmax(1), average="macro")
                if s > best: best, w = s, w2
    return w, best

def honest_check(P, y, k=5):
    gains = []
    for a, b in StratifiedKFold(k, shuffle=True, random_state=0).split(P, y):
        w, _ = tune_weights(P[a], y[a], rounds=2)
        base = f1_score(y[b], P[b].argmax(1), average="macro")
        tuned = f1_score(y[b], (P[b] * w).argmax(1), average="macro")
        gains.append(tuned - base)
    return np.mean(gains), np.std(gains)

w_lgb, s_lgb = tune_weights(oof_lgb, y)
print("LGB tuned OOF macro F1:", s_lgb, "weights:", np.round(w_lgb, 3))
print("Honest mean gain from tuning (+/- std):", honest_check(oof_lgb, y))

LGB tuned OOF macro F1: 0.3260682696765144 weights: [3.32  2.117 1.35  0.861 0.951]
Honest mean gain from tuning (+/- std): (np.float64(0.05284761739621775), np.float64(0.00437291482970556))


In [28]:
def save_sub(pred, w, path="submission.csv"):
    sub = pd.DataFrame({"id": te["id"],
                        "target_popularity": [classes[i] for i in (pred * w).argmax(1)]})
    assert len(sub) == len(te) and sub["id"].is_unique
    assert set(sub["target_popularity"]) <= set(classes)
    sub.to_csv(path, index=False)
    print(sub["target_popularity"].value_counts(normalize=True).sort_index())

save_sub(pred_lgb, w_lgb)

target_popularity
A    0.1136
B    0.1464
C    0.1788
D    0.2880
E    0.2732
Name: proportion, dtype: float64


In [29]:
import xgboost as xgb
from catboost import CatBoostClassifier

def run_xgb(seeds=SEEDS):
    oof = np.zeros((len(X), 5)); pred = np.zeros((len(Xt), 5))
    dt = xgb.DMatrix(Xt)
    for s in seeds:
        p = dict(objective="multi:softprob", num_class=5, eval_metric="mlogloss",
                 tree_method="hist", eta=0.04, max_depth=6, subsample=0.8,
                 colsample_bytree=0.7, min_child_weight=5, reg_lambda=5.0, seed=s)
        for f, (a, b) in enumerate(StratifiedKFold(N_FOLDS, shuffle=True, random_state=s).split(X, y)):
            da = xgb.DMatrix(X.iloc[a], y[a]); db = xgb.DMatrix(X.iloc[b], y[b])
            m = xgb.train(p, da, 3000, evals=[(db, "v")], early_stopping_rounds=100, verbose_eval=False)
            r = (0, m.best_iteration + 1)
            oof[b] += m.predict(db, iteration_range=r) / len(seeds)
            pred += m.predict(dt, iteration_range=r) / (N_FOLDS * len(seeds))
            print(f"xgb seed {s} fold {f}: iters={m.best_iteration}")
    return oof, pred

def run_cat(seeds=SEEDS):
    oof = np.zeros((len(X), 5)); pred = np.zeros((len(Xt), 5))
    for s in seeds:
        for f, (a, b) in enumerate(StratifiedKFold(N_FOLDS, shuffle=True, random_state=s).split(X, y)):
            m = CatBoostClassifier(loss_function="MultiClass", iterations=3000, learning_rate=0.06,
                                   depth=6, l2_leaf_reg=5, random_seed=s,
                                   early_stopping_rounds=100, verbose=0, thread_count=-1)
            m.fit(X.iloc[a], y[a], eval_set=(X.iloc[b], y[b]))
            oof[b] += m.predict_proba(X.iloc[b]) / len(seeds)
            pred += m.predict_proba(Xt) / (N_FOLDS * len(seeds))
            print(f"cat seed {s} fold {f}: iters={m.get_best_iteration()}")
    return oof, pred

oof_xgb, pred_xgb = run_xgb()
print("XGB raw OOF:", f1_score(y, oof_xgb.argmax(1), average="macro"))
oof_cat, pred_cat = run_cat()
print("CAT raw OOF:", f1_score(y, oof_cat.argmax(1), average="macro"))

xgb seed 7 fold 0: iters=260
xgb seed 7 fold 1: iters=268
xgb seed 7 fold 2: iters=234
xgb seed 7 fold 3: iters=360
xgb seed 7 fold 4: iters=240
XGB raw OOF: 0.26712910492927333
cat seed 7 fold 0: iters=509
cat seed 7 fold 1: iters=609
cat seed 7 fold 2: iters=553
cat seed 7 fold 3: iters=662
cat seed 7 fold 4: iters=396
CAT raw OOF: 0.2649062440580389


In [30]:
oof_bl = (oof_lgb + oof_xgb + oof_cat) / 3
pred_bl = (pred_lgb + pred_xgb + pred_cat) / 3
print("Blend raw OOF:", f1_score(y, oof_bl.argmax(1), average="macro"))

w_bl, s_bl = tune_weights(oof_bl, y)
print("Blend tuned OOF:", s_bl, "weights:", np.round(w_bl, 3))
print("Honest gain from tuning:", honest_check(oof_bl, y))

if s_bl > s_lgb:
    save_sub(pred_bl, w_bl)
else:
    print("Blend not better than LGB alone, keeping the baseline submission.")

Blend raw OOF: 0.26536434665963815
Blend tuned OOF: 0.3294448367401045 weights: [3.158 2.117 1.35  0.905 0.951]
Honest gain from tuning: (np.float64(0.06262128334797648), np.float64(0.00526000216219375))
target_popularity
A    0.0960
B    0.1492
C    0.1620
D    0.3320
E    0.2608
Name: proportion, dtype: float64
